<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Classification 3 - 1D ECG


## Import Packages

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
num_epochs = 100
log_interval = 10
in_channels_ = 1
num_segments_in_record = 100
segment_len = 3600
num_records = 48
num_classes = 17
batch_size = 16
learning_rate = 0.0004

print(f"num_epochs: {num_epochs}")
print(f"log_interval: {log_interval}")
print(f"in_channels_: {in_channels_}")
print(f"num_segments_in_record: {num_segments_in_record}")
print(f"segment_len: {segment_len}")
print(f"num_records: {num_records}")
print(f"num_classes: {num_classes}")
print(f"batch_size: {batch_size}")
print(f"learning_rate: {learning_rate}")

## Prepare Dataset

In [ ]:
import os
import scipy.io as scio
from sklearn.model_selection import train_test_split
import numpy as np

# Clone the repository to get the dataset
repo_url = "https://github.com/vigneashpandiyan/Pytorch-for-KTEK0070-Machine-Learning-in-Digital-Manufacturing-.git"
repo_name = "Pytorch-for-KTEK0070-Machine-Learning-in-Digital-Manufacturing-"
if not os.path.exists(repo_name):
    print(f"Cloning repository: {repo_url}")
    !git clone {repo_url}
else:
    print(f"Repository '{repo_name}' already exists. Skipping clone.")


base_path = './'
# Update the dataset_path to point to the cloned directory
dataset_path =  os.path.join(repo_name, 'Pytorch DL', 'Data', 'ECG Dataset')

classes = ['NSR', 'APB', 'AFL', 'AFIB', 'SVTA', 'WPW','PVC', 'Bigeminy', 'Trigeminy',
           'VT', 'IVR', 'VFL', 'Fusion', 'LBBBB', 'RBBBB', 'SDHB', 'PR']
ClassesNum = len(classes)

X = []
y = []
expected_sample_length = 3600

# Traverse through the dataset directory
for root, dirs, files in os.walk(dataset_path):
    for name in files:
        if name.endswith('.mat'): # Process only .mat files
            file_path = os.path.join(root, name)
            try:
                data_train = scio.loadmat(file_path)
                data_arr = data_train.get('val') # Get 'val' key which contains ECG data

                if data_arr is not None and data_arr.ndim == 2 and data_arr.shape[0] == 1 and data_arr.shape[1] == expected_sample_length:
                    # ECG data is typically (1, 3600) or similar, we need the 3600 length array
                    data_list = data_arr.tolist()
                    X.append(data_list[0]) # Append the 3600 length list

                    # Extract class label from the parent directory name (e.g., '01', '02')
                    parent_dir_name = os.path.basename(root)
                    class_num_str = parent_dir_name[0:2] # Take first two characters, assuming "XX_..."
                    try:
                        class_label = int(class_num_str) - 1
                        if 0 <= class_label < ClassesNum: # Validate class label
                            y.append(class_label)
                        else:
                            print(f"Warning: Invalid class label '{class_label}' derived from '{parent_dir_name}' for file {file_path}. Skipping.")
                            X.pop() # Remove the added data as class is invalid
                    except ValueError:
                        print(f"Warning: Could not parse class number from directory name '{parent_dir_name}' for file {file_path}. Skipping.")
                        X.pop() # Remove the added data as class parsing failed
                else:
                    print(f"Warning: Skipping file {file_path}. 'val' key not found, or data shape is not (1, {expected_sample_length}), or unexpected dimensions. Shape: {data_arr.shape if data_arr is not None else 'None'}")
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

X_np = np.array(X)
y_np = np.array(y)

print(f"Number of samples loaded: {len(X_np)}")
print(f"Shape of loaded X: {X_np.shape}, Shape of loaded y: {y_np.shape}")

# Apply normalization function
def normalization(data):
    _range = np.max(data) - np.min(data)
    # Avoid division by zero if all values are the same
    if _range == 0:
        return np.zeros_like(data)
    return (data - np.min(data)) / _range

# Apply normalization to each sample
if len(X_np) > 0:
    X_normalized = np.array([normalization(sample) for sample in X_np])
else:
    X_normalized = np.array([])


# Reshape X and y as expected by the original code
# The original code explicitly reshaped to (1000, 1, 3600) and (1000)
# Let's stick to that if the loaded data matches the expected count
if len(X_normalized) == 1000 and X_normalized.ndim == 2 and X_normalized.shape[1] == expected_sample_length:
    X_reshaped = X_normalized.reshape((1000, 1, expected_sample_length))
    y_reshaped = y_np.reshape((1000,))

    # Split into train and test sets
    # Added stratify for balanced classes split, and random_state for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y_reshaped, test_size=0.2, random_state=42, stratify=y_reshaped)
    print("\nDataset successfully prepared with 1000 samples:")
    print("X_train : ", len(X_train))
    print("X_test  : ", len(X_test))
    print("shape of X_train : ", np.shape(X_train[0]))
    print("shape of y_train : ", np.shape(y_train))
    print("shape of X_test : ", np.shape(X_test))
    print("shape of y_test : ", np.shape(y_test))
elif len(X_normalized) == 0:
    print("\nNo data available after processing. X_train, X_test, y_train, y_test are initialized as empty arrays.")
    X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])
else:
    print(f"\nError: Expected 1000 samples of length {expected_sample_length}, but loaded {len(X_normalized)} samples of shape {X_normalized.shape if X_normalized.ndim > 1 else 'N/A'}.")
    print("Please ensure the dataset contains the expected number and shape of samples. X_train, X_test, y_train, y_test are initialized as empty arrays.")
    X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])


In [ ]:
class Flatten(torch.nn.Module):
    def forward(self, x):
        batch_size = x.shape[0]
        return x.view(batch_size, -1)

class arrhythmia_classifier(nn.Module):
    def __init__(self, in_channels=in_channels_):
        super(arrhythmia_classifier, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,128,50,stride=3),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(kernel_size=2, stride=3),

            nn.Conv1d(128,32,7,stride=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2,stride=2),

            nn.Conv1d(32,32,10,stride=1),
            nn.ReLU(),

            nn.Conv1d(32,128,5,stride=2),
            nn.ReLU(),
            nn.MaxPool1d(2,stride=2),

            nn.Conv1d(128,256,15,stride=1),
            nn.ReLU(),
            nn.MaxPool1d(2,2),

            nn.Conv1d(256,512,5,stride=1),
            nn.Conv1d(512,128,3,stride=1),

            Flatten(),
            nn.Linear(in_features=1152, out_features=512),
            nn.ReLU(),
            nn.Dropout(p=.1),
            nn.Linear(in_features=512, out_features=17),
        )

    def forward(self, x, ex_features=None):
        return self.cnn(x)


def calc_next_len_conv1d(current_len=112500, kernel_size=16, stride=8, padding=0, dilation=1):
    return int(np.floor((current_len + 2 * padding - dilation * (kernel_size - 1) - 1) / stride + 1))

model = arrhythmia_classifier()

In [ ]:
print(model)

## Construct Loss and Optimizer

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0004, betas=(0.9, 0.999), eps=1e-08, weight_decay = 0.0, amsgrad = False)

## Train

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Calculate class distribution for the training set
unique_train, counts_train = np.unique(y_train, return_counts=True)
train_class_counts = dict(zip(unique_train, counts_train))

# Calculate class distribution for the test set
unique_test, counts_test = np.unique(y_test, return_counts=True)
test_class_counts = dict(zip(unique_test, counts_test))

print("Training set class distribution:", train_class_counts)
print("Test set class distribution:", test_class_counts)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))

# Plotting training set class distribution
ax1.bar(np.arange(len(classes)), [train_class_counts.get(i, 0) for i in range(len(classes))])
ax1.set_xticks(np.arange(len(classes)))
ax1.set_xticklabels(classes, rotation=45, ha='right')
ax1.set_xlabel('Class Name')
ax1.set_ylabel('Number of Samples')
ax1.set_title('Training Set Class Distribution')

# Plotting test set class distribution
ax2.bar(np.arange(len(classes)), [test_class_counts.get(i, 0) for i in range(len(classes))])
ax2.set_xticks(np.arange(len(classes)))
ax2.set_xticklabels(classes, rotation=45, ha='right')
ax2.set_xlabel('Class Name')
ax2.set_ylabel('Number of Samples')
ax2.set_title('Test Set Class Distribution')

plt.tight_layout()
plt.show()

In [ ]:
unique_train_labels = np.unique(y_train)

plt.figure(figsize=(20, 10))

# Using a colormap for distinct colors for each class
colors = plt.cm.get_cmap('tab20', len(classes))

for i, class_label in enumerate(unique_train_labels):
    # Find the index of the first occurrence of this class in y_train
    # Ensure y_train is directly comparable if it contains numpy.int64 types
    idx = np.where(y_train == class_label)[0][0]

    # Retrieve the corresponding ECG signal from X_train and reshape it
    signal = X_train[idx].reshape(-1)

    # Plot the signal with a distinct color and label
    plt.plot(signal, color=colors(i), label=f'Class {classes[class_label]}')

plt.title('Representative ECG Signals for Each Class')
plt.xlabel('Time Samples')
plt.ylabel('Normalized Amplitude')
plt.legend(loc='best', bbox_to_anchor=(1, 1))
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import time

# Initialize lists to store metrics
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
train_times = []
test_times = []

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Create TensorDatasets
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

total_samples = len(y_train)
class_weights = {}
for class_label, count in train_class_counts.items():
    # Calculate inverse frequency weight: total_samples / class_count
    class_weights[class_label] = total_samples / count

# Create a list of weights for each sample in y_train
sample_weights = [class_weights[label] for label in y_train]

print("Calculated Class Weights (inverse frequency):")
for label, weight in class_weights.items():
    print(f"Class {label} ({classes[label]}): {weight:.2f}")

print(f"\nLength of sample_weights: {len(sample_weights)}")
print(f"First 10 sample weights: {sample_weights[:10]}")

In [ ]:
from torch.utils.data import WeightedRandomSampler

# Convert sample_weights to a PyTorch tensor
sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.float)

# Create a WeightedRandomSampler
# num_samples should typically be the total number of samples in the dataset
# or an integer multiple of batch_size for a full epoch.
# replacement=True means samples are drawn with replacement.
weighted_sampler = WeightedRandomSampler(sample_weights_tensor,
                                         num_samples=len(sample_weights_tensor),
                                         replacement=True)

# Update the train_loader to use the weighted_sampler
# Note: When using a sampler, shuffle must be False.
train_loader = torch.utils.data.DataLoader(train_dataset,
                                         batch_size=batch_size,
                                         sampler=weighted_sampler,
                                         shuffle=False) # Shuffle must be False when using a sampler

print("WeightedRandomSampler created and train_loader updated successfully.")
print(f"Train loader now uses a WeightedRandomSampler with {len(weighted_sampler)} samples.")


In [ ]:

def train(epoch):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()

        # forward + backward + update
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, dim=1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

        if batch_idx % log_interval == (log_interval - 1): # print every log_interval batches
            avg_batch_loss = running_loss / log_interval
            print(f'[Epoch {epoch + 1}, Batch {batch_idx + 1:5d}] loss: {avg_batch_loss:.8f}')
            running_loss = 0.0

    end_time = time.time()
    epoch_train_loss = running_loss # The last remaining loss if any, or 0.0 if log_interval divides perfectly
    epoch_train_accuracy = 100 * correct / total
    epoch_time = end_time - start_time

    train_losses.append(epoch_train_loss) # Store the last batch's loss or overall average if adjusted
    train_accuracies.append(epoch_train_accuracy)
    train_times.append(epoch_time)

    print(f'Epoch {epoch + 1} , Train Accuracy: {epoch_train_accuracy:.2f}%, Time: {epoch_time:.2f}s')

def test():
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    test_loss = 0.0
    start_time = time.time()

    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            loss = criterion(outputs, labels) # Calculate loss on test set
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    end_time = time.time()
    avg_test_loss = test_loss / len(test_loader) # Average loss over the test set
    test_accuracy = 100 * correct / total
    epoch_time = end_time - start_time

    test_losses.append(avg_test_loss)
    test_accuracies.append(test_accuracy)
    test_times.append(epoch_time)

    print(f'Accuracy on test set: {test_accuracy:.2f} % - Test Loss: {avg_test_loss:.8f}, Time: {epoch_time:.2f}s')

print("Starting training...")
# Main training loop
for epoch in range(num_epochs):
    train(epoch)
    test()

print("Training complete.")

In [ ]:
plt.figure(figsize=(15, 10))

# Plotting Training and Test Loss
plt.subplot(2, 1, 1)
plt.plot(test_losses, label='Training Loss', color='blue')
plt.title('Training and Test Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Plotting Training and Test Accuracy
plt.subplot(2, 1, 2)
plt.plot(train_accuracies, label='Training Accuracy', color='blue')
plt.plot(test_accuracies, label='Test Accuracy', color='red')
plt.title('Training and Test Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# Regarding balancing classes in the test loader:
# It is generally not recommended to balance classes in the test set.
# The test set should reflect the true, real-world distribution of your data,
# which often means it will be imbalanced if the real-world data is.
# Balancing the test set can lead to an overly optimistic or misleading
# assessment of your model's performance in a production environment.

# Instead, class imbalance should primarily be addressed during the
# TRAINING phase (e.g., using weighted sampling in the DataLoader for training,
# oversampling minority classes, undersampling majority classes, or
# using class-weighted loss functions).

# For EVALUATION on imbalanced datasets, it's important to use appropriate metrics
# such as precision, recall, F1-score for each class, the confusion matrix,
# and ROC AUC curves (as already being generated in this notebook).

from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns

# Set model to evaluation mode
model.eval()

# Initialize lists to store true labels, predicted labels, and prediction probabilities
all_labels = []
all_preds = []
all_probs = []

# Ensure test_loader is defined from previous steps, or define it if not.
# Assuming test_loader is already defined based on prior execution state.
# If not, it would need to be created from X_test, y_test, and batch_size.
# For example:
# test_dataset = torch.utils.data.TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long())
# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        # If using GPU, move tensors to GPU
        # images = images.to(device)
        # labels = labels.to(device)

        outputs = model(images)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probabilities.cpu().numpy())

# Convert collected lists to numpy arrays
all_labels_np = np.array(all_labels)
all_preds_np = np.array(all_preds)
all_probs_np = np.array(all_probs)

print("Collected all labels, predictions, and probabilities from the test set.")
print(f"Shape of all_labels_np: {all_labels_np.shape}")
print(f"Shape of all_preds_np: {all_preds_np.shape}")
print(f"Shape of all_probs_np: {all_probs_np.shape}")

In [ ]:
cm = confusion_matrix(all_labels_np, all_preds_np)

# Normalize the confusion matrix by row (divide by the sum of each row)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.1%', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Normalized Confusion Matrix (Percentages)')
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))

# Plot ROC curve for each class
for i in range(num_classes):
    # Binarize the true labels for the current class (one-vs-rest)
    y_true_bin = (all_labels_np == i).astype(int)

    # Check if there are positive samples for this class in the test set
    # If not, skip plotting ROC for this class as it's not meaningful
    if np.sum(y_true_bin) == 0:
        print(f"Skipping ROC for class {classes[i]} (label {i}) due to no positive samples in test set.")
        continue

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_true_bin, all_probs_np[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f'ROC curve of class {classes[i]} (AUC = {roc_auc:.2f})')

# Plot the random classifier baseline
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (AUC = 0.50)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves for Each Class')
plt.legend(loc='lower right', bbox_to_anchor=(1.0, 0.0))
plt.grid(True)
plt.tight_layout()
plt.show()